# Treinar a voz da Nora no Piper (Google Colab)

Ajuste fino de uma voz do Piper com o conjunto gerado por `nora voz gerar-dataset`.
Guia completo: `docs/TREINAR_VOZ.md` no repositório da Nora.

**Antes de começar**

1. Compacte a pasta `data\dataset_voz\<nome>` (com `wavs\` e `metadata.csv`) em `<nome>.zip` e envie para o Google Drive, em `Meu Drive/Nora/`.
2. Aqui no Colab: **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)**.
3. Rode as células **em ordem**, de cima para baixo.

**Se o Colab desconectar:** abra de novo, rode as células 1 a 5 e a de treino de novo. Ele continua do último ponto salvo no Drive.

**Licenças:** o Piper (`piper1-gpl`, GPL-3.0) é baixado aqui, à parte, e nunca entra no repositório da Nora.
As vozes base do Piper costumam derivar de dados só para uso não comercial: a voz treinada é para **uso pessoal**
e fica fora do Git (`models/`).

## 1. Configuração

In [ ]:
NOME_VOZ = "nora_v1"  # mesmo nome usado no gerar-dataset
PASTA_DRIVE = "/content/drive/MyDrive/Nora"  # onde está o .zip; os checkpoints também ficam aqui
MAX_EPOCAS = 2000  # receita do Piper para ajuste fino com menos de 5 h de fala
LOTE = 16  # 16 cabe com folga na T4; 24 ou 32 se a GPU tiver mais memória
# Vazio = escolhe uma voz pt_BR "medium"; ou o caminho de um .ckpt do piper-checkpoints
VOZ_BASE = ""
SALVAR_NO_DRIVE_A_CADA_MIN = 15  # cópia do checkpoint para o Drive durante o treino
FRASE_DE_TESTE = "Oi! Hoje de manhã eu organizei a sua agenda. Você quer ouvir os compromissos?"

## 2. Drive e GPU

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Instalar o Piper (uns 5 minutos)

In [ ]:
import os

if not os.path.isdir("/content/piper1-gpl"):
    !apt-get -qq update && apt-get -qq install -y build-essential cmake ninja-build > /dev/null
    !git clone -q --depth 1 https://github.com/OHF-Voice/piper1-gpl.git /content/piper1-gpl
    %cd /content/piper1-gpl
    !pip -q install -e '.[train]'
    !./build_monotonic_align.sh
    !python3 setup.py build_ext --inplace
    %cd /content
print("Piper pronto.")

## 4. Conjunto de treino

In [ ]:
import pathlib
import shutil
import zipfile

import soundfile as sf

ZIP = pathlib.Path(PASTA_DRIVE) / f"{NOME_VOZ}.zip"
DADOS = pathlib.Path("/content/dataset")
shutil.rmtree(DADOS, ignore_errors=True)
with zipfile.ZipFile(ZIP) as arquivo_zip:
    arquivo_zip.extractall(DADOS)

META = next(DADOS.rglob("metadata.csv"))  # o zip pode ter uma pasta por fora
WAVS = META.parent / "wavs"
boas, faltando, taxas, segundos = [], [], set(), 0.0
for linha in META.read_text(encoding="utf-8").splitlines():
    if not linha.strip():
        continue
    ident, _, texto = linha.partition("|")
    wav = WAVS / (ident if ident.endswith(".wav") else f"{ident}.wav")
    if not wav.is_file() or not texto.strip():
        faltando.append(ident)
        continue
    info = sf.info(wav)
    taxas.add(info.samplerate)
    segundos += info.duration
    boas.append(f"{ident}|{texto.strip()}")

META_OK = DADOS / "metadata_ok.csv"
META_OK.write_text("\n".join(boas) + "\n", encoding="utf-8")
print(f"{len(boas)} frases, {segundos / 60:.1f} min de fala.")
if faltando:
    print(f"Ignoradas (sem áudio ou sem texto): {', '.join(faltando[:20])}")
assert taxas == {22050}, f"Os áudios precisam estar em 22.050 Hz; encontrei {taxas}."
if segundos < 5 * 60:
    print(
        "Atenção: menos de 5 minutos de fala. A voz pode errar palavras que não estão no conjunto."
    )
horas = segundos / 3600
print(f"Estimativa na T4: ~{(60 * horas + 2) * MAX_EPOCAS / 3600:.0f} h para {MAX_EPOCAS} épocas.")

## 5. Voz base (ponto de partida do ajuste fino)

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files

REPO = "rhasspy/piper-checkpoints"
if VOZ_BASE.startswith("/"):
    BASE = VOZ_BASE
else:
    ckpts = [a for a in list_repo_files(REPO, repo_type="dataset") if a.endswith(".ckpt")]
    pt_br = [a for a in ckpts if "pt_BR" in a and "/medium/" in a]
    print("Vozes pt_BR medium disponíveis:", *pt_br, sep="\n  ")
    escolhida = VOZ_BASE or (
        pt_br[0] if pt_br else next(a for a in ckpts if "en_US/lessac/medium" in a)
    )
    print("Usando:", escolhida)
    BASE = hf_hub_download(REPO, escolhida, repo_type="dataset")

## 6. Treinar

Pode levar horas. Deixe a aba aberta. A cada 15 minutos o último checkpoint é copiado para o Drive.
Para ouvir como está, pare esta célula (botão ■), rode a célula 7 e depois rode esta de novo: ela continua de onde parou.

In [ ]:
import glob
import shlex
import threading

LOCAL = "/content/treino"
SALVO = pathlib.Path(PASTA_DRIVE) / f"treino_{NOME_VOZ}"
SALVO.mkdir(parents=True, exist_ok=True)
CONFIG = f"{LOCAL}/config.json"
os.makedirs(LOCAL, exist_ok=True)


def ultimo_checkpoint():
    locais = sorted(
        glob.glob(f"{LOCAL}/lightning_logs/version_*/checkpoints/last.ckpt"), key=os.path.getmtime
    )
    if locais:
        return locais[-1]
    no_drive = SALVO / "last.ckpt"
    return str(no_drive) if no_drive.is_file() else None


def copiar_para_o_drive():
    origem = ultimo_checkpoint()
    if origem and not origem.startswith(str(SALVO)):
        shutil.copy(origem, SALVO / "last.ckpt.tmp")
        os.replace(SALVO / "last.ckpt.tmp", SALVO / "last.ckpt")
    if os.path.isfile(CONFIG):
        shutil.copy(CONFIG, SALVO / "config.json")


parar = threading.Event()


def copiar_sempre():
    while not parar.wait(SALVAR_NO_DRIVE_A_CADA_MIN * 60):
        try:
            copiar_para_o_drive()
        except OSError as erro:
            print("[drive] não copiou:", erro)


opcoes = {
    "--data.voice_name": NOME_VOZ,
    "--data.csv_path": META_OK,
    "--data.audio_dir": WAVS,
    "--model.sample_rate": 22050,
    "--data.espeak_voice": "pt-br",
    "--data.cache_dir": "/content/cache",
    "--data.config_path": CONFIG,
    "--data.batch_size": LOTE,
    "--trainer.max_epochs": MAX_EPOCAS,
    "--trainer.default_root_dir": LOCAL,
    "--model.mos_metric": "none",
}
comando = ["python3", "-m", "piper.train", "fit"]
for opcao, valor in opcoes.items():
    comando += [opcao, str(valor)]
retomar = ultimo_checkpoint()
if retomar:
    print("Continuando de", retomar)
    comando += ["--ckpt_path", retomar]
else:
    print("Começando do zero a partir da voz base.")
    comando += ["--model.warmstart_ckpt", BASE]

copiador = threading.Thread(target=copiar_sempre, daemon=True)
copiador.start()
try:
    get_ipython().system(shlex.join(comando))  # noqa: F821 (existe no Colab)
finally:
    parar.set()
    copiar_para_o_drive()
    print("Checkpoint salvo no Drive:", SALVO / "last.ckpt")

## 7. Exportar e ouvir

In [ ]:
import shlex
import wave

from IPython.display import Audio, display
from piper import PiperVoice

SAIDA = pathlib.Path("/content/saida")
SAIDA.mkdir(exist_ok=True)
NOME_ARQUIVO = f"pt_BR-{NOME_VOZ.replace('_', '')}-medium"
ONNX = SAIDA / f"{NOME_ARQUIVO}.onnx"

ckpt = ultimo_checkpoint()
print("Exportando", ckpt)
exportar = ["python3", "-m", "piper.train.export_onnx", "--checkpoint", ckpt, "--output-file", ONNX]
get_ipython().system(shlex.join(map(str, exportar)))  # noqa: F821 (existe no Colab)
config = CONFIG if os.path.isfile(CONFIG) else str(SALVO / "config.json")
shutil.copy(config, f"{ONNX}.json")

voz = PiperVoice.load(str(ONNX))
teste = SAIDA / "teste.wav"
with wave.open(str(teste), "wb") as arquivo_wav:
    voz.synthesize_wav(FRASE_DE_TESTE, arquivo_wav)
display(Audio(str(teste)))

## 8. Levar para a Nora

Quando gostar do resultado, rode esta célula: os dois arquivos vão para `Meu Drive/Nora/`.
No PC, copie-os para `B:\Nora\models\piper\` e, no `config\local.toml`:

```toml
[voz]
motor = "piper"

[voz.piper]
voz = "pt_BR-norav1-medium"   # o nome mostrado abaixo
```

In [ ]:
for arquivo in (ONNX, pathlib.Path(f"{ONNX}.json")):
    shutil.copy(arquivo, pathlib.Path(PASTA_DRIVE) / arquivo.name)
print("No Drive:", ONNX.name, "e", ONNX.name + ".json")
print(f'No local.toml: voz = "{NOME_ARQUIVO}"')